# Lesson 6.5 — Template Variables and Advanced Grafana Use Cases

## Learning Objectives
- Create template variables in Grafana using AppDynamics query types
- Compare metrics across multiple applications using variable substitution
- Use the `${application}` variable in Metrics queries
- Build a complete multi-app comparison dashboard

**Reference**: [AppDynamics Query Editor — Use Cases](https://grafana.com/docs/plugins/dlopes7-appdynamics-datasource/latest/query-editor/#use-cases)

## 1. Template Variables in Grafana

Grafana template variables work similarly to Dash Studio variables:
1. Define a variable with a query source
2. Reference it as `${variableName}` in panel queries
3. Users see a dropdown at the top of the dashboard

### Creating a template variable for AppDynamics:

**Variable: Application**
1. Dashboard Settings → Variables → Add variable
2. Name: `application`
3. Type: **Query**
4. Data source: AppDynamics
5. Query: `Applications` (this is the query type, not ADQL)
6. The plugin queries the Controller for all applications and populates the dropdown
7. Enable **Multi-value** to allow selecting multiple applications

**Variable: Transaction**
1. Add another variable: `transactionName`
2. Data source: AppDynamics (Analytics)
3. Query type: Analytics
4. ADQL: `SELECT distinct(transactionName) FROM transactions ORDER BY transactionName ASC`
5. This auto-populates with actual BT names from your data

In [ ]:
# Document all template variable configurations for a production Grafana dashboard

grafana_template_variables = [
    {
        "name": "application",
        "label": "Application",
        "type": "query",
        "datasource": "AppDynamics Production",
        "query_type": "Applications",
        "multi_value": True,
        "include_all": True,
        "description": "Populated from AppDynamics - lists all monitored applications",
        "used_in_queries": [
            "Metrics: Application = ${application}",
            "Analytics: WHERE applicationName IN (${application:doublequote})"
        ]
    },
    {
        "name": "transactionName",
        "label": "Transaction",
        "type": "query",
        "datasource": "AppDynamics Production",
        "query_type": "Analytics",
        "adql": "SELECT distinct(transactionName) FROM transactions ORDER BY transactionName ASC",
        "multi_value": True,
        "include_all": True,
        "description": "Populated from ADQL - lists all BTs that have data in the time range",
        "used_in_queries": [
            "Analytics: WHERE transactionName IN (${transactionName:doublequote})"
        ]
    },
    {
        "name": "tier",
        "label": "Tier",
        "type": "query",
        "datasource": "AppDynamics Production",
        "query_type": "Tiers",
        "application_var": "${application}",
        "multi_value": False,
        "description": "Populated from AppDynamics - lists tiers for selected application",
        "cascades_from": "application",
        "used_in_queries": [
            "Metrics: metric path tier segment",
            "Analytics: WHERE tierName = '${tier}'"
        ]
    },
    {
        "name": "userExperience",
        "label": "Experience",
        "type": "custom",
        "options": ["NORMAL", "SLOW", "VERY_SLOW", "ERROR"],
        "default": "SLOW,VERY_SLOW,ERROR",
        "multi_value": True,
        "description": "Static list of user experience values",
        "used_in_queries": [
            "Analytics: WHERE userExperience IN (${userExperience:doublequote})"
        ]
    }
]

print("=== Grafana Template Variable Configurations ===")
for v in grafana_template_variables:
    print(f"\nVariable: ${{{v['name']}}}")
    print(f"  Label      : {v['label']}")
    print(f"  Type       : {v['type']}")
    print(f"  Multi-value: {v['multi_value']}")
    if v['type'] == 'query':
        print(f"  Query type : {v['query_type']}")
        if 'adql' in v:
            print(f"  ADQL       : {v['adql']}")
    elif v['type'] == 'custom':
        print(f"  Options    : {v['options']}")
    print(f"  Description: {v['description']}")
    if 'cascades_from' in v:
        print(f"  Cascades from: ${{{v['cascades_from']}}} (updates when parent changes)")

## 2. Official Use Cases from Grafana Docs

These use cases are documented in the [AppDynamics query editor docs](https://grafana.com/docs/plugins/dlopes7-appdynamics-datasource/latest/query-editor/#use-cases):

### Monitor application response time:
1. Select **Metrics** query type
2. Select application from **Application** dropdown
3. Navigate to `Overall Application Performance|Average Response Time (ms)`
4. Set **Labels** to **Custom**: `{{app}} - Response Time`

### Display a single metric value (Stat panel):
1. Select **Metrics**
2. Navigate to `Overall Application Performance|Calls per Minute`
3. Set **Aggregation** to **Sum**
4. Toggle **Roll-up** ON
5. Use **Stat** or **Gauge** visualization

### Compare metrics across applications:
1. Create a template variable with the query `Applications`
2. Select **Metrics** query
3. Select `${application}` from the **Application** dropdown
4. Navigate to the desired metric path
5. When multiple applications are selected, the plugin generates **a separate query for each application** and displays them as individual series

### Analyze business transactions (Analytics):
```sql
SELECT transactionName, count(*) AS total, avg(responseTime) AS avgResponseTime
FROM transactions ORDER BY total DESC
```

### Identify slow transactions:
```sql
SELECT transactionName, responseTime, eventTimestamp FROM transactions
WHERE responseTime > 5000 ORDER BY responseTime DESC
```

### Monitor user experience distribution:
```sql
SELECT series(eventTimestamp, '5m'), userExperience, count(*) FROM transactions
```
Returns time series grouped by experience category — suitable for stacked bar or time series visualization.

In [ ]:
# Final production-ready dashboard definition
# This documents a complete Grafana dashboard for AppDynamics

production_dashboard = {
    "title": "AppDynamics — StoreFront Full-Stack Dashboard",
    "tags": ["appdynamics", "apm", "observability"],
    "variables": [
        {"name": "application",      "type": "Applications query"},
        {"name": "transactionName",   "type": "Analytics ADQL — distinct transactions"},
        {"name": "tier",             "type": "Tiers query (cascades from application)"},
        {"name": "userExperience",    "type": "Custom static list"},
    ],
    "rows": [
        {
            "title": "Application Overview",
            "panels": [
                {"type": "Stat",        "title": "Avg Response Time",    "query": "Metrics: Overall Application Performance|Average Response Time (ms), Roll-up ON"},
                {"type": "Stat",        "title": "Calls per Minute",     "query": "Metrics: Overall Application Performance|Calls per Minute, Sum, Roll-up ON"},
                {"type": "Stat",        "title": "Errors per Minute",    "query": "Metrics: Overall Application Performance|Errors per Minute, Roll-up ON"},
            ]
        },
        {
            "title": "Transaction Performance",
            "panels": [
                {"type": "Time Series", "title": "Response Time Trend",  "query": "Metrics: ... BT Performance wildcard"},
                {"type": "Time Series", "title": "User Experience Mix",  "query": "Analytics: series + userExperience"},
            ]
        },
        {
            "title": "Error Analysis",
            "panels": [
                {"type": "Table",       "title": "Top Slowest BTs",       "query": "Analytics: ADQL grouped by transactionName, ORDER BY maxRT"},
                {"type": "Pie Chart",   "title": "Errors by Country",    "query": "Analytics: ADQL country + count WHERE errorOccurred=true"},
            ]
        },
        {
            "title": "Health and Events",
            "panels": [
                {"type": "State Timeline", "title": "Health Rule Violations", "query": "Health query type"},
                {"type": "Annotation",     "title": "Deployments + Incidents", "query": "Events query type (annotation layer)"},
            ]
        },
        {
            "title": "Infrastructure",
            "panels": [
                {"type": "Time Series", "title": "CPU % by Node",   "query": "Metrics: Hardware Resources|CPU|%Busy — tier tier, node *"},
                {"type": "Time Series", "title": "JVM Heap by Node", "query": "Metrics: JVM|Heap|Current Usage (MB) — tier, node *"},
            ]
        }
    ]
}

print(f"Dashboard: {production_dashboard['title']}")
print(f"Tags: {production_dashboard['tags']}")
print()
print("Template Variables:")
for v in production_dashboard['variables']:
    print(f"  ${{{v['name']}}} — {v['type']}")
print()
print("Panels:")
for row in production_dashboard['rows']:
    print(f"  Row: {row['title']}")
    for panel in row['panels']:
        print(f"    [{panel['type']:15}] {panel['title']}")

In [ ]:
# EXERCISE 6.5: Build the multi-app comparison panel
#
# From the official docs:
# "When multiple applications are selected, the plugin generates a separate
# query for each application and displays them as individual series."
#
# TASK: Configure a Time Series panel that shows the Average Response Time
# for ALL selected applications simultaneously (multi-app comparison).

multi_app_panel = {
    "title": "Multi-App Response Time Comparison",
    "type": "timeseries",
    "variable_prerequisite": "${application} must be a multi-value variable backed by the 'Applications' query type",
    "query": {
        "query_type": "???",         # TODO: Metrics or Analytics?
        "application": "???",        # TODO: use variable
        "metric_path": "???",        # TODO: which metric to compare?
        "label_mode": "???",         # TODO: how to label each series?
        "label_template": "???",     # TODO: useful label showing app name
        "aggregation": "???",        # TODO: Value, Sum, Avg?
    },
    "expected_behavior": "When user selects StoreFront + PaymentService, two lines appear on the chart"
}

print("Multi-App Comparison Panel Configuration:")
print(f"  Title: {multi_app_panel['title']}")
print(f"  Prerequisite: {multi_app_panel['variable_prerequisite']}")
print(f"  Query type   : {multi_app_panel['query']['query_type']}")
print(f"  Application  : {multi_app_panel['query']['application']}")
print(f"  Metric path  : {multi_app_panel['query']['metric_path']}")
print(f"  Label mode   : {multi_app_panel['query']['label_mode']}")
print(f"  Label template: {multi_app_panel['query']['label_template']}")
print(f"  Aggregation  : {multi_app_panel['query']['aggregation']}")
print(f"  Expected     : {multi_app_panel['expected_behavior']}")

## Summary

- Grafana template variables for AppDynamics can be populated from `Applications`, `Tiers`, `Metric Names`, or Analytics ADQL queries
- Use `:doublequote` formatter for multi-value variables in ADQL `IN` clauses: `${var:doublequote}`
- When `${application}` is multi-valued and used in a Metrics query, the plugin generates one query per application and shows all as separate series
- The full observability stack: AppDynamics Metrics (historical trends) + Analytics ADQL (event-level detail) + Health (violations) + Events (annotations) — all in one Grafana dashboard

---

**Phase 6 Complete!** You are ready for the [Capstone Project](../capstone/capstone-project.ipynb).